# Destilação supervisionada — Conversa Pi + NVIDIA

Notebook experimental para usar um **professor remoto** via NVIDIA API e treinar os componentes do repositório [conversa-llm](https://github.com/lloupp/conversa-llm).

**Antes da coleta:** verifique que o modelo professor autoriza expressamente usar respostas para treinar modelos independentes; a mera disponibilidade pela API não garante esse direito. Não envie dados sensíveis ou chaves no prompt. Veja [NVIDIA_DISTILLATION.md](https://github.com/lloupp/conversa-llm/blob/feat/nvidia-supervised-distillation-20261015/docs/NVIDIA_DISTILLATION.md). Nenhum treino ocorre sem você executar este notebook no Colab.

In [ ]:
import os, subprocess, pathlib
REPO = pathlib.Path('/content/conversa-llm')
BRANCH = 'feat/nvidia-supervised-distillation-20261015'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
      'https://github.com/lloupp/conversa-llm.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['python','-m','pip','install','-q','-e','.[dev]'],check=True)
subprocess.run(['python','-m','pytest','-q','tests/test_distill_nvidia.py'],check=True)

In [ ]:
# Secrets (ícone de chave no Colab): NVIDIA_API_KEY com "Notebook access"
from google.colab import userdata
api_key = userdata.get('NVIDIA_API_KEY')
if not api_key or not api_key.strip():
    raise RuntimeError('Configure NVIDIA_API_KEY nos Secrets do Colab')
os.environ['NVIDIA_API_KEY'] = api_key.strip()
TEACHER = input('Modelo NVIDIA autorizado (ID exato): ').strip()
if not TEACHER:
    raise ValueError('Modelo não informado')
confirm = input('Verifiquei a licença e os termos; autorizam treino do meu modelo com as saídas? Digite CONFIRMO: ')
if confirm.strip() != 'CONFIRMO':
    raise RuntimeError('Permissão não confirmada; coleta bloqueada')
CALL_BUDGET = 10
OUT_DIR = 'data/distilled_nvidia'
subprocess.run(['python','scripts/distill_nvidia.py','--dry-run'],check=True)
subprocess.run(['python','scripts/distill_nvidia.py','--teacher',TEACHER,
                '--license-confirmed','--limit',str(CALL_BUDGET),
                '--out-dir',OUT_DIR],check=True)
for name in ('generation_train.jsonl','decision_train.jsonl','manifest.jsonl'):
    p=REPO/OUT_DIR/name
    print(name,len(p.read_text().splitlines()) if p.exists() else 0)

## Continuar seus pesos existentes (opcional)

O tokenizer **precisa ser exatamente o original**; caso contrário, não use o checkpoint. Sem checkpoint, o novo treino inicializa pesos do zero e não melhora automaticamente seu modelo anterior. Não há `--init-from` no treinador de System One atual.

In [ ]:
from google.colab import files
CONTINUE_EXISTING = False  # mude para True se quiser carregar os dois arquivos
CHECKPOINT = REPO/'checkpoints'/'checkpoint_original.pt'
TOKENIZER = REPO/'data'/'tokenizer_original.json'
if CONTINUE_EXISTING:
    print('Envie o checkpoint .pt e o tokenizer .json ORIGINAL')
    for name,content in files.upload().items():
        dest = CHECKPOINT if name.endswith('.pt') else TOKENIZER if name.endswith('.json') else None
        if dest is None: raise ValueError('Use somente .pt e .json')
        dest.parent.mkdir(parents=True,exist_ok=True)
        dest.write_bytes(content)
    if not CHECKPOINT.exists() or not TOKENIZER.exists():
        raise RuntimeError('Checkpoint e tokenizer original são obrigatórios')

In [ ]:
# Construir currículos base e, apenas em treino novo, gerar o tokenizer.
for cmd in [
    ['python','scripts/build_pi_curriculum.py'],
    ['python','scripts/build_pi_agent_curriculum.py'],
    ['python','scripts/build_decision_curriculum.py'],
]:
    subprocess.run(cmd,check=True)
GEN = REPO/OUT_DIR/'generation_train.jsonl'
DEC = REPO/OUT_DIR/'decision_train.jsonl'
if not CONTINUE_EXISTING:
    TOKENIZER = REPO/'data'/'pi_bpe.json'
    corpus=['data/pi_curriculum.jsonl','data/pi_agent_curriculum.jsonl']
    if GEN.exists() and GEN.stat().st_size: corpus.append(str(GEN))
    subprocess.run(['python','scripts/build_bpe_tokenizer.py','--data',*corpus,
                    '--out',str(TOKENIZER),'--vocab-size','768'],check=True)
print('Tokenizer:',TOKENIZER)

In [ ]:
# Treinar Conversa Pi. Se houver checkpoint, continuar pesos originais sem sobrescrevê-los.
import torch
GEN_OUT='checkpoints/conversa-nvidia-sft.pt'
corpus=['data/pi_curriculum.jsonl','data/pi_agent_curriculum.jsonl']
if GEN.exists() and GEN.stat().st_size: corpus.append(str(GEN))
command=['python','-m','conversa_llm.train','--tokenizer-file',str(TOKENIZER),
         '--data',*corpus,'--steps','100','--out',GEN_OUT]
if CONTINUE_EXISTING:
    config=torch.load(CHECKPOINT,map_location='cpu',weights_only=True)['config']
    command+=['--init-from',str(CHECKPOINT),'--d-model',str(config['d_model']),
              '--layers',str(config['n_layers']),'--heads',str(config['n_heads']),
              '--context',str(config['context_length']),
              '--batch-size','1','--grad-accum','8']
else:
    command+=['--profile','cpu-8gb-pi']
subprocess.run(command,check=True)

In [ ]:
# Treinar um NOVO classificador System One (não substitui checkpoint anterior)
DEC_OUT='checkpoints/decision-nvidia-sft.pt'
data=['data/decision_curriculum.jsonl']
if DEC.exists() and DEC.stat().st_size: data.append(str(DEC))
subprocess.run(['python','-m','conversa_llm.train_decision',
                '--tokenizer-file',str(TOKENIZER),'--data',*data,
                '--steps','100','--out',DEC_OUT],check=True)

In [ ]:
# Avaliar em benchmarks mantidos fora da coleta
for cmd in [
    ['python','-m','conversa_llm.eval_tools','--model',GEN_OUT,'--tokenizer-file',str(TOKENIZER)],
    ['python','-m','conversa_llm.eval_agent','--model',GEN_OUT,'--tokenizer-file',str(TOKENIZER)],
    ['python','-m','conversa_llm.eval_decision','--model',DEC_OUT,'--tokenizer-file',str(TOKENIZER),
     '--data','data/decision_benchmark.jsonl'],
]:
    result=subprocess.run(cmd,check=False)
    print('Avaliação',cmd[2], 'exit status',result.returncode)
print('Compare com seu baseline antes de usar em produção.')

In [ ]:
# Exportar resultados. Chave NVIDIA não é incluída.
import zipfile
from google.colab import files
archive=pathlib.Path('/content/conversa_distillation_results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as zip_out:
    for p in [REPO/GEN_OUT,REPO/DEC_OUT,REPO/OUT_DIR/'manifest.jsonl',
              REPO/OUT_DIR/'generation_train.jsonl',REPO/OUT_DIR/'decision_train.jsonl',TOKENIZER]:
        if p.exists(): zip_out.write(p,arcname=str(p.relative_to(REPO)))
print('Pacote criado:',archive)
files.download(str(archive))